# Module 2.1: Basic Tool Use with LLMs

<span class="badge blue">Agentic AI</span> <span class="badge amber">~15 min</span> <span class="badge mint">Hands-on</span>

## Learning Objectives

By completing this notebook, you will:
1. Understand how LLM function calling works under the hood
2. Implement your first tool with proper schema definition
3. Build a tool execution loop that handles LLM tool requests
4. Handle tool errors gracefully and return results to the LLM
5. Create tools with multiple parameters and validation

## Prerequisites

- Module 0 completed (API setup and tokenization)
- Understanding of JSON Schema basics
- Python functions and type hints

## Estimated Time: 60 minutes

---

## 1. Setup & Imports

We'll use the Anthropic SDK for tool calling demonstrations. Claude's tool use API provides clear, structured function calling capabilities.

In [ ]:
# Setup: Course styling and plot theme
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname("__file__"), "../../../../.."))

from resources.notebook_style import apply_course_theme
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()
print("Course theme applied.")

In [ ]:
# Install required packages
# !pip install anthropic openai

import json
import os
from typing import Dict, List, Any, Callable, Optional
from datetime import datetime
import anthropic

print("✓ Imports successful")

## 2. Understanding Tool Calling

**Tool calling (function calling) allows LLMs to request actions**. The flow is:

```
1. You define tools (functions) with schemas
2. LLM receives user query + available tools
3. LLM decides if it needs a tool
4. If yes: LLM returns tool_use request (with parameters)
5. You execute the tool in your code
6. You send tool result back to LLM
7. LLM generates final response using tool result
```

**Key Components**:
- **Tool Schema**: JSON description of the function (name, description, parameters)
- **Tool Execution**: Your code that runs when LLM requests the tool
- **Result Handling**: Sending tool output back to the LLM

In [ ]:
# Purpose: Define a simple tool schema
# Key Concept: Tools need clear descriptions for LLM to know when to use them

def create_calculator_tool() -> Dict:
    """
    Create a simple calculator tool definition.
    
    Returns
    -------
    dict
        Tool schema in Anthropic's format
    """
    return {
        "name": "calculator",
        "description": "Performs basic arithmetic operations (add, subtract, multiply, divide). Use this when the user asks for mathematical calculations.",
        "input_schema": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "enum": ["add", "subtract", "multiply", "divide"],
                    "description": "The mathematical operation to perform"
                },
                "a": {
                    "type": "number",
                    "description": "First number"
                },
                "b": {
                    "type": "number",
                    "description": "Second number"
                }
            },
            "required": ["operation", "a", "b"]
        }
    }

# Display the tool schema
calculator_tool = create_calculator_tool()
print("Calculator Tool Schema:")
print(json.dumps(calculator_tool, indent=2))

## 3. Implementing Tool Execution

Once the LLM requests a tool, we need to execute it and return results. The tool implementation is separate from the schema—the schema is for the LLM, the implementation is your Python code.

In [ ]:
# Purpose: Implement the calculator tool logic
# Key Concept: Tool implementations should handle errors gracefully

def execute_calculator(operation: str, a: float, b: float) -> Dict[str, Any]:
    """
    Execute calculator operation.
    
    Parameters
    ----------
    operation : str
        Mathematical operation (add, subtract, multiply, divide)
    a : float
        First number
    b : float
        Second number
    
    Returns
    -------
    dict
        Result with success status and value or error message
    """
    try:
        # Perform the operation
        if operation == "add":
            result = a + b
        elif operation == "subtract":
            result = a - b
        elif operation == "multiply":
            result = a * b
        elif operation == "divide":
            if b == 0:
                return {
                    "success": False,
                    "error": "Cannot divide by zero"
                }
            result = a / b
        else:
            return {
                "success": False,
                "error": f"Unknown operation: {operation}"
            }
        
        return {
            "success": True,
            "result": result,
            "operation": operation,
            "inputs": {"a": a, "b": b}
        }
    
    except Exception as e:
        return {
            "success": False,
            "error": str(e)
        }

# Test the tool implementation
print("Testing calculator implementation:")
print(execute_calculator("add", 5, 3))
print(execute_calculator("divide", 10, 2))
print(execute_calculator("divide", 10, 0))  # Error case

## 4. Building a Tool-Using Agent

Now we'll connect everything: the LLM, tool schema, and tool execution. This creates a complete agent loop.

**The Agent Loop**:
1. Send user message + tool definitions to LLM
2. Check LLM response for tool_use requests
3. Execute requested tools
4. Send tool results back to LLM
5. Get final response from LLM

In [ ]:
# Purpose: Create a complete agent with tool calling capability
# Key Concept: Agent loops handle back-and-forth between LLM and tool execution

class SimpleToolAgent:
    """Basic agent that can use tools."""
    
    def __init__(self, api_key: Optional[str] = None):
        """
        Initialize agent with API client.
        
        Parameters
        ----------
        api_key : str, optional
            Anthropic API key (uses ANTHROPIC_API_KEY env var if not provided)
        """
        self.client = anthropic.Anthropic(
            api_key=api_key or os.environ.get("ANTHROPIC_API_KEY")
        )
        self.tools = {}
        self.tool_schemas = []
    
    def register_tool(
        self,
        schema: Dict,
        implementation: Callable
    ) -> None:
        """
        Register a tool with its schema and implementation.
        
        Parameters
        ----------
        schema : dict
            Tool schema (Anthropic format)
        implementation : callable
            Function that executes the tool
        """
        tool_name = schema["name"]
        self.tools[tool_name] = implementation
        self.tool_schemas.append(schema)
        print(f"✓ Registered tool: {tool_name}")
    
    def execute_tool(self, tool_name: str, tool_input: Dict) -> Dict:
        """
        Execute a tool by name with given input.
        
        Parameters
        ----------
        tool_name : str
            Name of the tool to execute
        tool_input : dict
            Parameters for the tool
        
        Returns
        -------
        dict
            Tool execution result
        """
        if tool_name not in self.tools:
            return {"success": False, "error": f"Unknown tool: {tool_name}"}
        
        try:
            # Execute the tool function with provided parameters
            result = self.tools[tool_name](**tool_input)
            return result
        except Exception as e:
            return {"success": False, "error": f"Tool execution failed: {str(e)}"}
    
    def run(self, user_message: str, max_iterations: int = 5) -> str:
        """
        Run the agent with a user message.
        
        Parameters
        ----------
        user_message : str
            User's input message
        max_iterations : int
            Maximum tool use iterations to prevent infinite loops
        
        Returns
        -------
        str
            Final response from the agent
        """
        messages = [{"role": "user", "content": user_message}]
        
        for iteration in range(max_iterations):
            print(f"\n--- Iteration {iteration + 1} ---")
            
            # Call Claude with tools
            response = self.client.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=1024,
                tools=self.tool_schemas,
                messages=messages
            )
            
            print(f"Stop reason: {response.stop_reason}")
            
            # Check if Claude wants to use a tool
            if response.stop_reason == "tool_use":
                # Find the tool use block
                tool_use_block = None
                for block in response.content:
                    if block.type == "tool_use":
                        tool_use_block = block
                        break
                
                if tool_use_block:
                    print(f"Tool requested: {tool_use_block.name}")
                    print(f"Parameters: {tool_use_block.input}")
                    
                    # Execute the tool
                    tool_result = self.execute_tool(
                        tool_use_block.name,
                        tool_use_block.input
                    )
                    
                    print(f"Tool result: {tool_result}")
                    
                    # Add assistant's response and tool result to conversation
                    messages.append({
                        "role": "assistant",
                        "content": response.content
                    })
                    
                    messages.append({
                        "role": "user",
                        "content": [
                            {
                                "type": "tool_result",
                                "tool_use_id": tool_use_block.id,
                                "content": json.dumps(tool_result)
                            }
                        ]
                    })
                    
                    # Continue loop to get final response
                    continue
            
            # No tool use - extract text response
            final_response = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_response += block.text
            
            return final_response
        
        return "Max iterations reached without final response"

# Create and configure agent
agent = SimpleToolAgent()
agent.register_tool(
    schema=create_calculator_tool(),
    implementation=execute_calculator
)

print("\n" + "="*60)
print("Agent ready! Tools registered:")
for tool_name in agent.tools.keys():
    print(f"  - {tool_name}")

## 5. Testing the Tool-Using Agent

Let's test our agent with queries that require tool use.

In [ ]:
# Purpose: Test agent with mathematical queries
# Key Concept: LLM should automatically decide when to use tools

# Test 1: Simple calculation
print("TEST 1: Simple Addition")
print("="*60)
response1 = agent.run("What is 156 plus 289?")
print(f"\nFinal Response: {response1}")

print("\n" + "="*60 + "\n")

# Test 2: Multiple operations
print("TEST 2: Complex Calculation")
print("="*60)
response2 = agent.run("Calculate 50 times 12, then divide the result by 4")
print(f"\nFinal Response: {response2}")

## Hands-On Exercises

Build your own tools and extend the agent's capabilities.

### Exercise 2.1.1: Create a Time Tool

**Task**: Implement a tool that returns the current time in different formats.

**Expected Output**: Tool should support "iso", "unix", and "human" time formats.

**Hints**:
<details>
<summary>Hint 1</summary>
Use datetime.now() and its format methods (.isoformat(), .timestamp(), .strftime()).
</details>

<details>
<summary>Hint 2</summary>
Schema should have a "format" parameter with enum of allowed values.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def create_time_tool() -> Dict:
    """
    Create schema for time tool.
    
    Returns
    -------
    dict
        Tool schema
    """
    # TODO: Define the tool schema
    pass

def execute_time(format: str = "iso") -> Dict[str, Any]:
    """
    Get current time in specified format.
    
    Parameters
    ----------
    format : str
        Time format: 'iso', 'unix', or 'human'
    
    Returns
    -------
    dict
        Current time in requested format
    """
    # TODO: Implement time tool
    pass

exercise_2_1_1_schema = create_time_tool  # Don't change
exercise_2_1_1_impl = execute_time  # Don't change

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2_1_1():
    # Test schema
    schema = exercise_2_1_1_schema()
    assert isinstance(schema, dict), "Schema should be a dictionary"
    assert "name" in schema, "Schema needs 'name' field"
    assert "input_schema" in schema, "Schema needs 'input_schema' field"
    
    # Test implementation
    result_iso = exercise_2_1_1_impl("iso")
    assert isinstance(result_iso, dict), "Should return a dictionary"
    assert "success" in result_iso or "time" in result_iso or "result" in result_iso, \
        "Result should indicate success or contain time"
    
    result_unix = exercise_2_1_1_impl("unix")
    assert isinstance(result_unix, dict), "Should return a dictionary"
    
    result_human = exercise_2_1_1_impl("human")
    assert isinstance(result_human, dict), "Should return a dictionary"
    
    print("✓ Exercise 2.1.1 passed!")

test_exercise_2_1_1()

### Exercise 2.1.2: String Manipulation Tool

**Task**: Create a tool that performs string operations (uppercase, lowercase, reverse, length).

**Expected Output**: Tool accepts text and operation type, returns transformed text.

**Hints**:
<details>
<summary>Hint 1</summary>
Use string methods: .upper(), .lower(), [::-1], len()
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def create_string_tool() -> Dict:
    """
    Create schema for string manipulation tool.
    """
    # TODO: Define schema with 'text' and 'operation' parameters
    pass

def execute_string(text: str, operation: str) -> Dict[str, Any]:
    """
    Perform string operation.
    
    Parameters
    ----------
    text : str
        Input text
    operation : str
        One of: 'uppercase', 'lowercase', 'reverse', 'length'
    """
    # TODO: Implement string operations
    pass

exercise_2_1_2_schema = create_string_tool  # Don't change
exercise_2_1_2_impl = execute_string  # Don't change

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2_1_2():
    schema = exercise_2_1_2_schema()
    assert "name" in schema and "input_schema" in schema, "Invalid schema structure"
    
    # Test operations
    result_upper = exercise_2_1_2_impl("hello", "uppercase")
    assert isinstance(result_upper, dict), "Should return dict"
    
    result_reverse = exercise_2_1_2_impl("hello", "reverse")
    assert isinstance(result_reverse, dict), "Should return dict"
    
    result_length = exercise_2_1_2_impl("hello", "length")
    assert isinstance(result_length, dict), "Should return dict"
    
    print("✓ Exercise 2.1.2 passed!")

test_exercise_2_1_2()

### Exercise 2.1.3: Multi-Tool Agent

**Task**: Create an agent that has both calculator AND your time tool registered.

**Expected Output**: Agent can handle queries requiring either tool.

**Hints**:
<details>
<summary>Hint 1</summary>
Create a new SimpleToolAgent instance and register both tools.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def create_multi_tool_agent() -> SimpleToolAgent:
    """
    Create agent with calculator and time tools.
    
    Returns
    -------
    SimpleToolAgent
        Configured agent
    """
    # TODO: Create agent and register both tools
    pass

exercise_2_1_3_answer = create_multi_tool_agent  # Don't change

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2_1_3():
    agent = exercise_2_1_3_answer()
    assert isinstance(agent, SimpleToolAgent), "Should return SimpleToolAgent instance"
    assert len(agent.tools) >= 2, "Should have at least 2 tools registered"
    assert len(agent.tool_schemas) >= 2, "Should have at least 2 tool schemas"
    
    print("✓ Exercise 2.1.3 passed!")
    print(f"  Agent has {len(agent.tools)} tools: {list(agent.tools.keys())}")

test_exercise_2_1_3()

## Solutions

Reference implementations for all exercises.

In [ ]:
# SOLUTION 2.1.1: Time Tool

def create_time_tool_solution() -> Dict:
    return {
        "name": "get_time",
        "description": "Get the current time in different formats. Use when user asks for current time or date.",
        "input_schema": {
            "type": "object",
            "properties": {
                "format": {
                    "type": "string",
                    "enum": ["iso", "unix", "human"],
                    "description": "Time format: iso (ISO 8601), unix (timestamp), human (readable)"
                }
            },
            "required": ["format"]
        }
    }

def execute_time_solution(format: str = "iso") -> Dict[str, Any]:
    try:
        now = datetime.now()
        
        if format == "iso":
            time_str = now.isoformat()
        elif format == "unix":
            time_str = str(int(now.timestamp()))
        elif format == "human":
            time_str = now.strftime("%Y-%m-%d %H:%M:%S")
        else:
            return {"success": False, "error": f"Unknown format: {format}"}
        
        return {
            "success": True,
            "time": time_str,
            "format": format
        }
    except Exception as e:
        return {"success": False, "error": str(e)}

# SOLUTION 2.1.2: String Tool

def create_string_tool_solution() -> Dict:
    return {
        "name": "string_manipulator",
        "description": "Perform string operations like uppercase, lowercase, reverse, or get length.",
        "input_schema": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string",
                    "description": "The text to manipulate"
                },
                "operation": {
                    "type": "string",
                    "enum": ["uppercase", "lowercase", "reverse", "length"],
                    "description": "The operation to perform"
                }
            },
            "required": ["text", "operation"]
        }
    }

def execute_string_solution(text: str, operation: str) -> Dict[str, Any]:
    try:
        if operation == "uppercase":
            result = text.upper()
        elif operation == "lowercase":
            result = text.lower()
        elif operation == "reverse":
            result = text[::-1]
        elif operation == "length":
            result = len(text)
        else:
            return {"success": False, "error": f"Unknown operation: {operation}"}
        
        return {
            "success": True,
            "result": result,
            "operation": operation,
            "original": text
        }
    except Exception as e:
        return {"success": False, "error": str(e)}

# SOLUTION 2.1.3: Multi-Tool Agent

def create_multi_tool_agent_solution() -> SimpleToolAgent:
    agent = SimpleToolAgent()
    agent.register_tool(create_calculator_tool(), execute_calculator)
    agent.register_tool(create_time_tool_solution(), execute_time_solution)
    return agent

## Summary

**Key Takeaways**:

1. **Tool Schema**: Clear descriptions help LLMs decide when to use tools
2. **Separation of Concerns**: Schema (for LLM) vs Implementation (your code) are separate
3. **Error Handling**: Always return structured responses with success/error indicators
4. **Agent Loop**: Tool calling requires back-and-forth between LLM and your code
5. **Multiple Tools**: LLMs can choose from many tools based on the task

**What's Next**:
- Module 2.2: Building multi-tool agents with complex workflows
- Module 2.3: Tool security and sandboxing
- Module 4: ReAct agents with tool use

**Additional Resources**:
- [Anthropic Tool Use Guide](https://docs.anthropic.com/claude/docs/tool-use)
- [OpenAI Function Calling](https://platform.openai.com/docs/guides/function-calling)

---

You've built your first tool-using agent! This is the foundation for all advanced agent capabilities.